# scFoundation + GEARS (Frozen) — GSE152988

Trains GEARS with scFoundation as a **frozen** feature extractor on GSE152988 CRISPRi and CRISPRa data.

---

## Before you start — checklist

1. **Runtime → Change runtime type → A100 GPU** (40 GB VRAM required for frozen mode)
2. Upload your data files to Google Drive under `MyDrive/566_project/GSE152988/`:
   - `TianKampmann2021_CRISPRi.h5ad`
   - `TianKampmann2021_CRISPRa.h5ad`
3. Run cells **top to bottom**. Do not skip.
4. If the session dies mid-training, re-run from **Cell 8** — the preprocessing outputs are cached to Drive and will be reused.

---

## Steps overview
| Cell | What it does |
|------|--------------|
| 1 | Check GPU — must be A100 |
| 2 | Mount Google Drive |
| 3 | Install dependencies |
| 4 | Clone scFoundation repo |
| 5 | Download pretrained model weights |
| 6 | Config — update paths here if needed |
| 7 | Preprocess + split data (cached to Drive) |
| 8 | Train CRISPRi: scFoundation + GEARS frozen |
| 9 | Train CRISPRa: scFoundation + GEARS frozen |
| 10 | Collect and display results |

In [ ]:
# Cell 1: Check GPU
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
gpu_info = result.stdout.strip()
print(f"GPU: {gpu_info}")

if 'A100' not in gpu_info:
    print()
    print("WARNING: Not on A100. You may hit OOM with scFoundation.")
    print("Options:")
    print("  - Go to Runtime > Change runtime type > select A100")
    print("  - Or reduce BATCH_SIZE to 4 in Cell 6 and try anyway (slower, may still OOM)")
else:
    print("A100 confirmed — good to go.")

In [ ]:
# Cell 2: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted at /content/drive")

In [ ]:
# Cell 3: Install dependencies
# This takes 3-5 minutes on first run
!pip install -q torch==1.13.1+cu117 torchvision==0.14.1+cu117 \
    --extra-index-url https://download.pytorch.org/whl/cu117

!pip install -q torch-geometric
!pip install -q torch-scatter torch-sparse torch-cluster \
    -f https://data.pyg.org/whl/torch-1.13.1+cu117.html

!pip install -q cell-gears scanpy anndata scipy scikit-learn huggingface_hub

print("\nInstallation complete.")

In [ ]:
# Cell 4: Clone scFoundation repo
import os

REPO_DIR = "/content/scFoundation"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/biomap-research/scFoundation.git {REPO_DIR}
    print("Repo cloned.")
else:
    print("Repo already exists — skipping clone.")

# Install any repo-specific requirements
req_path = os.path.join(REPO_DIR, "requirements.txt")
if os.path.exists(req_path):
    !pip install -q -r {req_path}

print(f"\nRepo contents:")
!ls {REPO_DIR}

In [ ]:
# Cell 5: Download pretrained scFoundation weights
# ~1-2 GB download; cached to Drive so you don't re-download if session dies
import os
from huggingface_hub import hf_hub_download

WEIGHTS_DRIVE = "/content/drive/MyDrive/566_project/scFoundation_weights"
WEIGHTS_LOCAL = "/content/scFoundation/model/models"
os.makedirs(WEIGHTS_DRIVE, exist_ok=True)
os.makedirs(WEIGHTS_LOCAL, exist_ok=True)

ckpt_drive = os.path.join(WEIGHTS_DRIVE, "models.ckpt")
ckpt_local = os.path.join(WEIGHTS_LOCAL, "models.ckpt")

if not os.path.exists(ckpt_drive):
    print("Downloading weights from HuggingFace (~1-2 GB)...")
    hf_hub_download(
        repo_id="biomap-research/scFoundation",
        filename="models.ckpt",
        local_dir=WEIGHTS_DRIVE
    )
    print("Download complete. Cached to Drive.")
else:
    print("Weights already on Drive — skipping download.")

# Symlink/copy to local path expected by scFoundation scripts
if not os.path.exists(ckpt_local):
    !cp {ckpt_drive} {ckpt_local}
    print(f"Copied weights to {ckpt_local}")
else:
    print(f"Weights already at {ckpt_local}")

# Also copy the gene index file if needed
gene_index_candidates = [
    f"{REPO_DIR}/model/OS_scRNA_gene_index.19264.tsv",
    f"{REPO_DIR}/OS_scRNA_gene_index.19264.tsv",
]
for p in gene_index_candidates:
    if os.path.exists(p):
        print(f"Gene index found: {p}")
        GENE_INDEX_PATH = p
        break
else:
    print("WARNING: Gene index file not found — check repo structure")
    !find {REPO_DIR} -name '*.tsv' 2>/dev/null

In [ ]:
# Cell 6: Config — update paths here if your Drive layout is different

# ── Input data (on Drive) ─────────────────────────────────────────────────────
CRISPR_I_H5AD = "/content/drive/MyDrive/566_project/GSE152988/TianKampmann2021_CRISPRi.h5ad"
CRISPR_A_H5AD = "/content/drive/MyDrive/566_project/GSE152988/TianKampmann2021_CRISPRa.h5ad"

# ── scFoundation paths ────────────────────────────────────────────────────────
SCF_REPO      = "/content/scFoundation"
SCF_CKPT      = "/content/scFoundation/model/models/models.ckpt"
GEARS_SCRIPT  = f"{SCF_REPO}/GEARS/main.py"   # adjust if repo layout differs

# ── Output (written to Drive so results survive session death) ────────────────
RESULT_BASE   = "/content/drive/MyDrive/566_project/results/scfoundation_gears_frozen"

# ── Split cache (avoids re-preprocessing on session restart) ─────────────────
SPLIT_CACHE   = "/content/drive/MyDrive/566_project/split_cache"

# ── Training hyperparameters ──────────────────────────────────────────────────
EPOCHS        = 20
BATCH_SIZE    = 8    # 8 is safe on A100 40GB; try 16 if memory allows
LR            = 1e-4
SEED          = 42
VAL_FRAC      = 0.10
TEST_FRAC     = 0.10
MIN_CELLS     = 30

import os
os.makedirs(RESULT_BASE, exist_ok=True)
os.makedirs(SPLIT_CACHE, exist_ok=True)

# Verify data files exist
for path, label in [(CRISPR_I_H5AD, "CRISPRi"), (CRISPR_A_H5AD, "CRISPRa"), (SCF_CKPT, "model ckpt")]:
    status = "FOUND" if os.path.exists(path) else "MISSING"
    print(f"  {label}: {status} — {path}")

# Detect GEARS main.py location
if not os.path.exists(GEARS_SCRIPT):
    print("\nGEARS main.py not found at expected path — searching...")
    result = !find {SCF_REPO} -name 'main.py' 2>/dev/null
    for r in result:
        print(f"  Found: {r}")
    print("Update GEARS_SCRIPT above with the correct path.")
else:
    print(f"\nGEARS main.py: FOUND")

In [ ]:
# Cell 7: Preprocess and split data
# Results cached to Drive — safe to re-run after session restart
import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import warnings
warnings.filterwarnings("ignore")

def preprocess_and_split(h5ad_path, label, cache_dir, split_cache):
    """
    Load h5ad, normalize, split by condition (zero-shot style),
    and save train/val/test h5ads. Uses cache if already done.
    """
    train_out = os.path.join(split_cache, f"{label}_train.h5ad")
    val_out   = os.path.join(split_cache, f"{label}_val.h5ad")
    test_out  = os.path.join(split_cache, f"{label}_test.h5ad")

    if os.path.exists(train_out) and os.path.exists(val_out) and os.path.exists(test_out):
        print(f"[{label}] Split cache found — loading from Drive (skipping preprocessing)")
        return train_out, val_out, test_out

    print(f"\n[{label}] Loading {h5ad_path}...")
    adata = ad.read_h5ad(h5ad_path)
    print(f"  Shape: {adata.n_obs:,} cells x {adata.n_vars:,} genes")

    # Store raw counts
    adata.layers["counts"] = adata.X.copy()

    # Normalize
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)

    # Ensure condition column in GEARS format (GENE+ctrl or ctrl)
    if "perturbation" in adata.obs.columns and "condition" not in adata.obs.columns:
        adata.obs["condition"] = adata.obs["perturbation"].apply(
            lambda g: "ctrl" if g in ("control", "ctrl") else f"{g}+ctrl"
        )
    assert "condition" in adata.obs.columns, \
        "Need a 'condition' or 'perturbation' column in obs"

    adata.var["gene_name"] = adata.var_names.tolist()
    adata.obs["cell_type"] = "iPSC-induced-neuron"

    # Zero-shot split: hold out entire perturbation conditions for val/test
    np.random.seed(SEED)
    counts   = adata.obs["condition"].value_counts()
    eligible = [c for c in counts.index if c != "ctrl" and counts[c] >= MIN_CELLS]
    np.random.shuffle(eligible)

    n       = len(eligible)
    n_val   = max(1, int(n * VAL_FRAC))
    n_test  = max(1, int(n * TEST_FRAC))
    n_train = n - n_val - n_test

    train_conds = set(eligible[:n_train]) | {"ctrl"}
    val_conds   = set(eligible[n_train:n_train + n_val])
    test_conds  = set(eligible[n_train + n_val:])

    adata_train = adata[adata.obs["condition"].isin(train_conds)].copy()
    adata_val   = adata[adata.obs["condition"].isin(val_conds)].copy()
    adata_test  = adata[adata.obs["condition"].isin(test_conds)].copy()

    print(f"  Split: {n_train} train / {n_val} val / {n_test} test perturbations")
    print(f"  Cells — train: {adata_train.n_obs:,} | val: {adata_val.n_obs:,} | test: {adata_test.n_obs:,}")

    print(f"  Saving splits to Drive cache...")
    adata_train.write_h5ad(train_out)
    adata_val.write_h5ad(val_out)
    adata_test.write_h5ad(test_out)
    print(f"  Cached.")

    return train_out, val_out, test_out


# Run for both modalities
crispri_train, crispri_val, crispri_test = preprocess_and_split(
    CRISPR_I_H5AD, "CRISPRi", RESULT_BASE, SPLIT_CACHE)

crispra_train, crispra_val, crispra_test = preprocess_and_split(
    CRISPR_A_H5AD, "CRISPRa", RESULT_BASE, SPLIT_CACHE)

print("\nPreprocessing complete.")

In [ ]:
# Cell 8: Train scFoundation + GEARS (frozen) — CRISPRi
# Runtime: ~4-10 hours on A100 depending on dataset size and epochs
# Log is saved to Drive — if session dies, check the log to see how far it got
import os, subprocess

result_i = os.path.join(RESULT_BASE, "CRISPRi")
os.makedirs(result_i, exist_ok=True)

log_i = os.path.join(result_i, "train.log")

# Check if training is already done
done_flag = os.path.join(result_i, "DONE")
if os.path.exists(done_flag):
    print("CRISPRi training already complete — skipping. Delete DONE file to retrain.")
else:
    print(f"Starting CRISPRi training ({EPOCHS} epochs, batch_size={BATCH_SIZE})...")
    print(f"Log: {log_i}")
    print("This will take several hours. Monitor the log file for progress.\n")

    import sys
    sys.path.insert(0, SCF_REPO)
    gears_dir = os.path.dirname(GEARS_SCRIPT)

    cmd = [
        sys.executable, GEARS_SCRIPT,
        "--data_path",             crispri_train,
        "--val_data_path",         crispri_val,
        "--test_data_path",        crispri_test,
        "--model_type",            "maeautobin",
        "--bin_set",               "autobin_resolution_append",
        "--finetune_method",       "frozen",
        "--singlecell_model_path", SCF_CKPT,
        "--batch_size",            str(BATCH_SIZE),
        "--epochs",                str(EPOCHS),
        "--lr",                    str(LR),
        "--seed",                  str(SEED),
        "--result_dir",            result_i,
    ]

    print("Command:", " ".join(cmd))
    print("-" * 60)

    with open(log_i, "w") as log_f:
        proc = subprocess.Popen(
            cmd,
            cwd=gears_dir,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1
        )
        for line in proc.stdout:
            print(line, end="")       # stream to notebook
            log_f.write(line)         # also write to Drive log
            log_f.flush()

    proc.wait()
    if proc.returncode == 0:
        open(done_flag, "w").close()  # mark as done
        print("\nCRISPRi training complete.")
    else:
        print(f"\nTraining exited with code {proc.returncode}. Check log: {log_i}")

In [ ]:
# Cell 9: Train scFoundation + GEARS (frozen) — CRISPRa
import os, subprocess, sys

result_a = os.path.join(RESULT_BASE, "CRISPRa")
os.makedirs(result_a, exist_ok=True)

log_a = os.path.join(result_a, "train.log")
done_flag_a = os.path.join(result_a, "DONE")

if os.path.exists(done_flag_a):
    print("CRISPRa training already complete — skipping. Delete DONE file to retrain.")
else:
    print(f"Starting CRISPRa training ({EPOCHS} epochs, batch_size={BATCH_SIZE})...")
    print(f"Log: {log_a}")
    print("This will take several hours. Monitor the log file for progress.\n")

    gears_dir = os.path.dirname(GEARS_SCRIPT)

    cmd = [
        sys.executable, GEARS_SCRIPT,
        "--data_path",             crispra_train,
        "--val_data_path",         crispra_val,
        "--test_data_path",        crispra_test,
        "--model_type",            "maeautobin",
        "--bin_set",               "autobin_resolution_append",
        "--finetune_method",       "frozen",
        "--singlecell_model_path", SCF_CKPT,
        "--batch_size",            str(BATCH_SIZE),
        "--epochs",                str(EPOCHS),
        "--lr",                    str(LR),
        "--seed",                  str(SEED),
        "--result_dir",            result_a,
    ]

    print("Command:", " ".join(cmd))
    print("-" * 60)

    with open(log_a, "w") as log_f:
        proc = subprocess.Popen(
            cmd,
            cwd=gears_dir,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1
        )
        for line in proc.stdout:
            print(line, end="")
            log_f.write(line)
            log_f.flush()

    proc.wait()
    if proc.returncode == 0:
        open(done_flag_a, "w").close()
        print("\nCRISPRa training complete.")
    else:
        print(f"\nTraining exited with code {proc.returncode}. Check log: {log_a}")

In [ ]:
# Cell 10: Collect and display results
import json, os

print("=" * 65)
print("  scFoundation + GEARS (Frozen) — Results Summary")
print("=" * 65)

metric_keys = [
    "pearson_delta_top20_deg",
    "pearson_delta_da_markers",
    "sign_accuracy_top20_deg",
    "pearson_top20_deg",
    "pearson_all_genes",
    "mse",
]

all_results = {}
for label, res_dir in [("CRISPRi", os.path.join(RESULT_BASE, "CRISPRi")),
                        ("CRISPRa", os.path.join(RESULT_BASE, "CRISPRa"))]:
    # scFoundation+GEARS main.py typically writes metrics.json
    # Try both common output names
    for fname in ["metrics.json", "gears_metrics.json", "test_metrics.json"]:
        fpath = os.path.join(res_dir, fname)
        if os.path.exists(fpath):
            with open(fpath) as f:
                metrics = json.load(f)
            all_results[label] = metrics
            print(f"\n[{label}] ({fname})")
            for k, v in metrics.items():
                print(f"  {k}: {v}")
            break
    else:
        print(f"\n[{label}] No metrics file found yet in {res_dir}")
        print(f"  Check the log: {os.path.join(res_dir, 'train.log')}")
        # List what is there
        if os.path.exists(res_dir):
            files = os.listdir(res_dir)
            print(f"  Files present: {files}")

# Side-by-side comparison (if both ran)
if len(all_results) == 2:
    print("\n" + "=" * 65)
    print(f"  {'Metric':<35} {'CRISPRi':>12} {'CRISPRa':>12}")
    print(f"  {'-'*59}")
    all_keys = set(list(all_results["CRISPRi"].keys()) + list(all_results["CRISPRa"].keys()))
    for k in sorted(all_keys):
        vi = all_results["CRISPRi"].get(k, "—")
        va = all_results["CRISPRa"].get(k, "—")
        print(f"  {k:<35} {str(vi):>12} {str(va):>12}")

    # Save combined
    combined_path = os.path.join(RESULT_BASE, "combined_summary.json")
    with open(combined_path, "w") as f:
        json.dump(all_results, f, indent=2)
    print(f"\nCombined results saved: {combined_path}")

## Session recovery guide

If your Colab session dies mid-training:

1. Start a new session and **request A100** again
2. Re-run **Cell 1** through **Cell 6** (fast — installs and mounts Drive)
3. Re-run **Cell 7** — preprocessing is cached to Drive, will be skipped
4. Re-run **Cell 8 / Cell 9** — check the `train.log` on Drive first to see how far training got

If the `DONE` flag file exists in the result folder, training was completed successfully and that modality will be skipped.

## Troubleshooting

| Error | Fix |
|-------|-----|
| `CUDA out of memory` | Lower `BATCH_SIZE` to 4 in Cell 6, re-run |
| `GEARS_SCRIPT not found` | Check `!find /content/scFoundation -name main.py` output in Cell 6, update path |
| `ModuleNotFoundError` | Re-run Cell 3 (installs); restart runtime then re-run from Cell 1 |
| Training stuck at epoch 1 | Normal — scFoundation embedding extraction is slow for the first batch; give it 10-15 min |
| `models.ckpt` not found | Re-run Cell 5; check Drive path `566_project/scFoundation_weights/` |